In [1]:
import mlflow
# import mlflow.sklearn
import optuna

from optuna.integration.mlflow import MLflowCallback
from optuna.pruners import MedianPruner

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    OneHotEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [3]:
# =========================================================
# Columns
# =========================================================

nomod_columns = [
    'HasCrCard',
    'IsActiveMember',
    'Complain'
]

dummyfy_columns = [
    'Card Type',
    'NumOfProducts',
    'Geography',
    'Gender'
]

norm_std_columns = [
    'Balance',
    'Point Earned',
    'CreditScore',
    'Age',
    'Tenure',
    'Satisfaction Score'
]

In [4]:
# =========================================================
# Config
# =========================================================

RANDOM_STATE = 42
N_SPLITS = 5

EXPERIMENT_NAME = "customer-churn-optuna"

engineer features.
find shap_values to determine most important features

set optuna and mlflow.

In [5]:
# =========================================================
# Build Pipeline
# =========================================================

def build_pipeline(trial):

    # -----------------------------
    # Preprocessing
    # -----------------------------

    scaler_name = trial.suggest_categorical(
        'scaler',
        ['std', 'minmax', 'robust']
    )

    encoder_name = trial.suggest_categorical(
        'encoder',
        ['drop_first', 'no_drop']
    )

    scalers = {
        'std': StandardScaler(),
        'minmax': MinMaxScaler(),
        'robust': RobustScaler()
    }

    scaler = scalers[scaler_name]

    encoder = OneHotEncoder(
        handle_unknown='ignore',
        drop='first' if encoder_name == 'drop_first' else None
    )

    preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, dummyfy_columns),
        ('num', scaler, norm_std_columns),
        ('pass', 'passthrough', nomod_columns)
    ],
    remainder='drop'   # explicit is better than implicit
)

    # -----------------------------
    # Model Selection
    # -----------------------------

    model_name = trial.suggest_categorical(
        'model',
        ['lr', 'dt', 'rf']
    )

    if model_name == 'lr':

        model = LogisticRegression(
            C=trial.suggest_float(
                'lr_C',
                1e-3,
                10,
                log=True
            ),
            solver='lbfgs',
            max_iter=2000,
            random_state=RANDOM_STATE
        )

    elif model_name == 'dt':

        model = DecisionTreeClassifier(
            max_depth=trial.suggest_int(
                'dt_max_depth',
                2,
                20
            ),
            min_samples_split=trial.suggest_int(
                'dt_min_samples_split',
                2,
                20
            ),
            min_samples_leaf=trial.suggest_int(
                'dt_min_samples_leaf',
                1,
                10
            ),
            random_state=RANDOM_STATE
        )

    else:

        model = RandomForestClassifier(
            n_estimators=trial.suggest_int(
                'rf_n_estimators',
                100,
                500
            ),
            max_depth=trial.suggest_int(
                'rf_max_depth',
                2,
                20
            ),
            min_samples_split=trial.suggest_int(
                'rf_min_samples_split',
                2,
                20
            ),
            min_samples_leaf=trial.suggest_int(
                'rf_min_samples_leaf',
                1,
                10
            ),
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    # -----------------------------
    # Full Pipeline
    # -----------------------------

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    return pipeline

In [6]:
# =========================================================
# Objective Function
# =========================================================

def objective(trial):

    pipeline = build_pipeline(trial)

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1
    )

    mean_auc = np.mean(scores)

    # --------------------------------
    # Report intermediate value
    # (for pruning)
    # --------------------------------

    trial.report(mean_auc, step=0)

    if trial.should_prune():
        raise optuna.TrialPruned()

    return mean_auc

In [7]:
# =========================================================
# MLflow Setup
# =========================================================

mlflow.set_experiment(EXPERIMENT_NAME)

mlflc = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="auc",
    create_experiment=False,
    mlflow_kwargs={"nested": True}
)


# =========================================================
# Study
# =========================================================

study = optuna.create_study(
    direction='maximize',
    pruner=MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=0
    )
)

2026/05/14 15:12:58 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/14 15:12:58 INFO mlflow.store.db.utils: Updating database tables
2026/05/14 15:12:59 INFO mlflow.tracking.fluent: Experiment with name 'customer-churn-optuna' does not exist. Creating a new experiment.
/var/folders/bv/50x24wc545x5mclk_t88ryrc0000gn/T/ipykernel_16953/989846483.py:7: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflc = MLflowCallback(
[I 2026-05-14 15:12:59,292] A new study created in memory with name: no-name-a180bb96-09cc-466a-92c4-07b9df91487e


In [8]:
def get_feature_names(preprocessor):
    # Works only after fitting
    feature_names = preprocessor.get_feature_names_out()

    # sklearn returns names like:
    # "cat__Geography_France", "num__Balance", etc.
    return feature_names

In [9]:
# =========================================================
# Run Optimization
# =========================================================

with mlflow.start_run(run_name="optuna_search"):

    study.optimize(
        objective,
        n_trials=500,
        callbacks=[mlflc]
    )

    # --------------------------------
    # Best Metrics
    # --------------------------------

    mlflow.log_metric(
        "best_auc",
        study.best_value
    )

    # --------------------------------
    # Best Params
    # --------------------------------

    mlflow.log_params(study.best_params)

    # --------------------------------
    # Train Best Pipeline
    # --------------------------------

    best_pipeline = build_pipeline(study.best_trial)

    best_pipeline.fit(X_train, y_train)

    preprocessor = best_pipeline.named_steps['preprocessing']

    # BEFORE columns
    input_features = dummyfy_columns + norm_std_columns + nomod_columns

    # AFTER columns
    output_features = get_feature_names(preprocessor)

    mlflow.log_param("n_input_features", len(input_features))
    mlflow.log_param("n_output_features", len(output_features))
    mlflow.log_dict({
        "input_features": input_features,
        "output_features": output_features
    }, "feature_schema.json")

    # --------------------------------
    # Save Best Model
    # --------------------------------

    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        name="best_model",
        #artifact_path="best_model", # artifact_path deprecated soon
        serialization_format="skops"
    )

[I 2026-05-14 15:13:01,260] Trial 0 finished with value: 0.9987351433867626 and parameters: {'scaler': 'minmax', 'encoder': 'drop_first', 'model': 'dt', 'dt_max_depth': 15, 'dt_min_samples_split': 4, 'dt_min_samples_leaf': 3}. Best is trial 0 with value: 0.9987351433867626.
[I 2026-05-14 15:13:02,456] Trial 1 finished with value: 0.9994292353332656 and parameters: {'scaler': 'minmax', 'encoder': 'no_drop', 'model': 'lr', 'lr_C': 1.7392618109762268}. Best is trial 1 with value: 0.9994292353332656.
[I 2026-05-14 15:13:03,872] Trial 2 finished with value: 0.9994187073947488 and parameters: {'scaler': 'std', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 168, 'rf_max_depth': 18, 'rf_min_samples_split': 19, 'rf_min_samples_leaf': 6}. Best is trial 1 with value: 0.9994292353332656.
[I 2026-05-14 15:13:05,027] Trial 3 finished with value: 0.9987351433867626 and parameters: {'scaler': 'robust', 'encoder': 'no_drop', 'model': 'dt', 'dt_max_depth': 5, 'dt_min_samples_split': 10, 'dt_

In [10]:
# =========================================================
# Results
# =========================================================

print("\nBest ROC-AUC:")
print(study.best_value)

print("\nBest Params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")


Best ROC-AUC:
0.999626258182652

Best Params:
scaler: std
encoder: no_drop
model: lr
lr_C: 9.993146949745231


MLflow Experiment
    └── Optuna Study
            └── Trial Loop
                    ├── Choose hyperparameters (Bayesian optimization)
                    ├── Build preprocessing + model pipeline
                    ├── Run Cross Validation
                    ├── Compute ROC-AUC
                    ├── Report metric to Optuna
                    ├── Optuna decides:
                    │       ├── continue
                    │       └── prune trial
                    └── MLflow logs params + metrics
            └── Best trial selected
    └── Train best model on full training data
    └── Save model to MLflow